# Milestone 2 — Data Preprocessing
**Folder:** Feature_Engineering/preprocessing.ipynb
**Input:** Raw data (318 columns, 386,570 rows)
**Output:** Clean data saved to Data/processed/transactions_processed.parquet

## Steps
1. Load raw data
2. Check shape & column types
3. Drop unusable columns
4. Handle missing values
5. Convert numeric columns stored as strings
6. Extract date & time features
7. Encode categorical columns
8. Final check & save

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")

✅ Libraries imported


In [4]:
# Load all datasets
transactions = pd.read_parquet('../Data/raw/stg_transactions_features.parquet')
accounts     = pd.read_csv('../Data/raw/accounts.csv')
customers    = pd.read_csv('../Data/raw/customers.csv')
devices      = pd.read_csv('../Data/raw/devices.csv')
wallets      = pd.read_csv('../Data/raw/wallets.csv')

print("Transactions:", transactions.shape)
print("Accounts:",     accounts.shape)
print("Customers:",    customers.shape)
print("Devices:",      devices.shape)
print("Wallets:",      wallets.shape)

Transactions: (386570, 318)
Accounts: (5065, 10)
Customers: (3400, 44)
Devices: (5002, 10)
Wallets: (900, 12)


## Step 1 — Shape & Column Type Check
Verify raw data before making any changes.

In [5]:
print("=" * 50)
print("RAW DATASET OVERVIEW")
print("=" * 50)
print(f"Rows    : {transactions.shape[0]:,}")
print(f"Columns : {transactions.shape[1]}")
print()

print("Column types:")
print(transactions.dtypes.value_counts())
print()

# Feature group counts
rule_cols     = [c for c in transactions.columns if c.startswith('rule_')]
sender_cols   = [c for c in transactions.columns if c.startswith('sender_')]
receiver_cols = [c for c in transactions.columns if c.startswith('receiver_')]
ip_cols       = [c for c in transactions.columns if c.startswith('ip_flag')]

print("Feature groups in raw data:")
print(f"  Rule-based features  : {len(rule_cols)}")
print(f"  Sender velocity      : {len(sender_cols)}")
print(f"  Receiver velocity    : {len(receiver_cols)}")
print(f"  IP flag features     : {len(ip_cols)}")

RAW DATASET OVERVIEW
Rows    : 386,570
Columns : 318

Column types:
int64             175
str                84
float64            51
datetime64[us]      7
Int64               1
Name: count, dtype: int64

Feature groups in raw data:
  Rule-based features  : 127
  Sender velocity      : 48
  Receiver velocity    : 28
  IP flag features     : 8


## Step 2 — Drop Unusable Columns

### Why we drop these:
| Category | Examples | Reason |
|---|---|---|
| Unique IDs | transaction_id, session_id | Every row different — no pattern |
| PII | pan, aadhaar, mobile_number | Cannot use in ML model |
| Names | customer_name, counterparty_name | Free text, no value |
| Addresses | address_* columns | Free text, too high cardinality |
| High null datetime | account_wallet_inoperative_date (99.9% null) | Too much missing |
| High cardinality str | merchant_id (133,431 unique), beneficiary_wallet_id_vpa (50,032 unique) | Too many unique values |
| Single value | nationality (1 unique) | Zero information |

In [6]:
# ── 1. Unique ID columns ──────────────────────────────────────────────────
id_cols = [
    'transaction_id',                # 386,570 unique
    'session_id',                    # 386,570 unique
    'typology_group_id',             # 19,062 unique
    'customer_cif_id',               # 3,300 unique — identifier not feature
    'customer_branch_ifsc_code',     # 3,293 unique
    'counterparty_branch_ifsc_swift',# 3,305 unique
    'sender_cust_id_for_rollup',     # 3,300 unique
    'wallet_account_id',             # 858 unique
    'escrow_account_linked',         # 858 unique
    'device_id_fingerprint',         # 4,851 unique
    'ip_address',                    # 4,852 unique
    'rules_triggered',               # 2,250 unique combinations
]

# ── 2. PII columns ────────────────────────────────────────────────────────
pii_cols = [
    'pan',
    'aadhaar_number',
    'mobile_number',
    'email_id',
    'customer_name',
    'counterparty_name',
    'father_spouse_name',
    'name_beneficial_owners',
    'cif_beneficial_owners',
    'identification_proof_doc_no',
    'entity_identification_proof_doc_no',
]

# ── 3. Address free text ──────────────────────────────────────────────────
address_cols = [
    'address_registered_office',
    'address_place_of_business',
    'address_beneficial_owners',
    'address_individual_customer',
]

# ── 4. High cardinality string columns ───────────────────────────────────
high_card_cols = [
    'merchant_id',                      # 133,431 unique
    'beneficiary_wallet_id_vpa',        # 50,032 unique
    'load_source_account_card_details', # 6,671 unique
    'customer_account_number',          # 4,790 unique
    'counterparty_account_number',      # 4,825 unique
]

# ── 5. High null datetime columns ────────────────────────────────────────
high_null_cols = [
    'account_wallet_inoperative_date',  # 385,944 nulls (99.9%)
    'date_of_incorporation',            # 341,466 nulls (88.3%)
]

# ── 6. Zero information columns ───────────────────────────────────────────
zero_info_cols = [
    'nationality',  # only 1 unique value — tells nothing
]

# ── Combine and drop ──────────────────────────────────────────────────────
all_drop = (id_cols + pii_cols + address_cols +
            high_card_cols + high_null_cols + zero_info_cols)

# Only drop what actually exists in columns
all_drop_existing = [c for c in all_drop if c in transactions.columns]

before = transactions.shape[1]
transactions = transactions.drop(columns=all_drop_existing)
after = transactions.shape[1]

print(f"Columns BEFORE drop : {before}")
print(f"Columns dropped     : {len(all_drop_existing)}")
print(f"Columns AFTER drop  : {after}")
print()
print("Breakdown:")
print(f"  ID columns         : {len([c for c in id_cols if c in all_drop_existing])}")
print(f"  PII columns        : {len([c for c in pii_cols if c in all_drop_existing])}")
print(f"  Address columns    : {len([c for c in address_cols if c in all_drop_existing])}")
print(f"  High cardinality   : {len([c for c in high_card_cols if c in all_drop_existing])}")
print(f"  High null datetime : {len([c for c in high_null_cols if c in all_drop_existing])}")
print(f"  Zero info columns  : {len([c for c in zero_info_cols if c in all_drop_existing])}")

Columns BEFORE drop : 318
Columns dropped     : 35
Columns AFTER drop  : 283

Breakdown:
  ID columns         : 12
  PII columns        : 11
  Address columns    : 4
  High cardinality   : 5
  High null datetime : 2
  Zero info columns  : 1


## Step 3 — Handle Missing Values

Missing columns in raw data:
| Column | Null Count | % Missing | Action |
|---|---|---|---|
| account_wallet_inoperative_date | 385,944 | 99.9% | Already dropped |
| date_of_incorporation | 341,466 | 88.3% | Already dropped |
| date_of_birth | 45,104 | 11.7% | Fill with median date |
| professional_experience_years | 45,104 | 11.7% | Fill with median |

In [18]:
# Check missing before
print("Missing values BEFORE:")
missing = transactions.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing if len(missing) > 0 else "  None found")
print()

# ── Numeric columns → fill with median ───────────────────────────────────
num_cols = transactions.select_dtypes(include=['number']).columns
transactions[num_cols] = transactions[num_cols].fillna(
    transactions[num_cols].median())

# ── Datetime columns → fill with median date ─────────────────────────────
dt_cols = transactions.select_dtypes(include=['datetime']).columns
for col in dt_cols:
    if transactions[col].isnull().sum() > 0:
        median_date = transactions[col].dropna().median()
        transactions[col] = transactions[col].fillna(median_date)
        print(f"  Filled {col} with median: {median_date.date()}")

# ── String columns → fill with 'Unknown' ─────────────────────────────────
str_cols = transactions.select_dtypes(include=['str', 'object']).columns
transactions[str_cols] = transactions[str_cols].fillna('Unknown')

# Verify
total_missing = transactions.isnull().sum().sum()
print(f"\nMissing values AFTER : {total_missing}")
print("✅ No missing values remaining")

Missing values BEFORE:
date_of_birth                    45104
professional_experience_years    45104
dtype: int64

  Filled date_of_birth with median: 1985-03-23

Missing values AFTER : 0
✅ No missing values remaining


## Step 4 — Convert Numeric Columns Stored as Strings

Two columns look like numbers but are stored as strings:
- `wallet_balance_before` — 12,166 unique values like "1500.00"
- `wallet_balance_after`  — 9,882 unique values like "1200.50"

These need to be converted to float64 so the model can use them.

In [19]:
numeric_str_cols = ['wallet_balance_before', 'wallet_balance_after']

for col in numeric_str_cols:
    if col in transactions.columns:
        transactions[col] = pd.to_numeric(
            transactions[col], errors='coerce')
        # Fill any NaN created during conversion with median
        transactions[col] = transactions[col].fillna(
            transactions[col].median())
        print(f"  ✅ {col} → float64 | "
              f"min={transactions[col].min():.2f} | "
              f"max={transactions[col].max():.2f}")

print()
print(f"Shape still: {transactions.shape}")

  ✅ wallet_balance_before → float64 | min=1.85 | max=199963.58
  ✅ wallet_balance_after → float64 | min=0.00 | max=4940642.05

Shape still: (386570, 283)


## Step 5 — Extract Date & Time Features

### From `timestamp` (string HH:MM:SS):
- `hour` — time of day (0–23)
- `is_night` — 1 if hour is 10pm–5am (high risk window)

### From `datestamp` (datetime):
- `day_of_week` — Monday=0, Sunday=6
- `month` — 1–12
- `is_weekend` — 1 if Saturday or Sunday

### From `customer_cif_creation_date`:
- `customer_account_age_days` — how old is the customer's account

### From `account_wallet_opening_date`:
- `wallet_age_days` — how long the wallet has been open

### From `kyc_update_date`:
- `days_since_kyc_update` — stale KYC is a risk signal

### From `date_of_birth`:
- `customer_age_years` — age of customer

In [20]:
reference_date = pd.Timestamp('2025-12-01')

# ── From timestamp (string HH:MM:SS) ─────────────────────────────────────
transactions['hour'] = pd.to_datetime(
    transactions['timestamp'],
    format='%H:%M:%S',
    errors='coerce').dt.hour

transactions['is_night'] = (
    (transactions['hour'] >= 22) |
    (transactions['hour'] <= 5)).astype(int)

# ── From datestamp (already datetime) ────────────────────────────────────
transactions['day_of_week'] = transactions['datestamp'].dt.dayofweek
transactions['month']       = transactions['datestamp'].dt.month
transactions['is_weekend']  = (
    transactions['day_of_week'] >= 5).astype(int)

# ── From customer_cif_creation_date ──────────────────────────────────────
transactions['customer_account_age_days'] = (
    reference_date - transactions['customer_cif_creation_date']
).dt.days

# ── From account_wallet_opening_date ─────────────────────────────────────
transactions['wallet_age_days'] = (
    reference_date - transactions['account_wallet_opening_date']
).dt.days

# ── From kyc_update_date ──────────────────────────────────────────────────
transactions['days_since_kyc_update'] = (
    reference_date - transactions['kyc_update_date']
).dt.days

# ── From date_of_birth ────────────────────────────────────────────────────
transactions['customer_age_years'] = (
    reference_date - transactions['date_of_birth']
).dt.days // 365

# ── Drop original datetime columns after extraction ───────────────────────
drop_dt = ['timestamp', 'datestamp', 'customer_cif_creation_date',
           'account_wallet_opening_date', 'kyc_update_date', 'date_of_birth']
drop_dt_existing = [c for c in drop_dt if c in transactions.columns]
transactions = transactions.drop(columns=drop_dt_existing)

# Preview new features
new_feats = ['hour', 'is_night', 'day_of_week', 'month', 'is_weekend',
             'customer_account_age_days', 'wallet_age_days',
             'days_since_kyc_update', 'customer_age_years']

print("✅ Date & time features created:")
print(transactions[new_feats].head(5).to_string())
print()
print(f"Night transactions   : {transactions['is_night'].sum():,}")
print(f"Weekend transactions : {transactions['is_weekend'].sum():,}")
print(f"Shape after          : {transactions.shape}")

✅ Date & time features created:
   hour  is_night  day_of_week  month  is_weekend  customer_account_age_days  wallet_age_days  days_since_kyc_update  customer_age_years
0     8         0            0     12           0                        992              939                    883                  49
1    22         1            2     12           0                        992              939                    883                  49
2    17         0            3     12           0                        992              939                    883                  49
3    15         0            4     12           0                        992              939                    883                  49
4     6         0            5     12           1                        992              939                    883                  49

Night transactions   : 34,724
Weekend transactions : 110,158
Shape after          : (386570, 286)


## Step 6 — Encode Categorical (String) Columns

Converting remaining string columns to numbers using LabelEncoder.
Target columns `is_aml` and `aml_typology` are kept as-is for now.

In [21]:
# These are ALL remaining str columns that are meaningful to encode
# (verified from your exact dataset)
encode_cols = [
    # Transaction info
    'currency',                       # 6 unique
    'transaction_type_dr_cr',         # 2 unique
    'transaction_mode_channel_bank',  # 13 unique
    'cash_flag',                      # 2 unique
    'transaction_type_ppi',           # 7 unique
    'transaction_mode_channel_ppi',   # 6 unique
    'transaction_status',             # 4 unique
    'refund_chargeback_flag',         # 2 unique
    'source_of_funds_wallet',         # 6 unique
    'load_instrument_type',           # 5 unique

    # Wallet & limits
    'account_wallet_status',          # 4 unique
    'wallet_kyc_category',            # 3 unique
    'transaction_limit_per_txn',      # 3 unique
    'daily_transaction_limit',        # 3 unique
    'monthly_transaction_limit',      # 3 unique
    'annual_transaction_limit',       # 3 unique
    'maximum_wallet_balance_limit',   # 3 unique

    # Customer profile flags
    'non_face_to_face_flag',          # 2 unique
    'pep_flag',                       # 2 unique
    'hni_flag',                       # 2 unique
    'minor_flag',                     # 2 unique
    'vkyc_flag',                      # 2 unique
    'tax_residency',                  # 2 unique
    'residency',                      # 2 unique

    # Customer profile categories
    'customer_type',                  # 2 unique
    'customer_entity_type',           # 10 unique
    'account_category',               # 7 unique
    'account_type',                   # 5 unique
    'customer_current_risk_score',    # 3 unique
    'source_of_funds',                # 14 unique
    'citizenship',                    # 15 unique
    'customer_occupation_industry',   # 27 unique
    'beneficial_owner_types',         # 4 unique
    'passive_nfe',                    # 3 unique

    # Location & device
    'sender_country_code',            # 14 unique
    'receiver_country_code',          # 20 unique
    'merchant_name',                  # 43 unique
    'merchant_category_code',         # 25 unique
    'merchant_location',              # 61 unique
    'geo_location_city_country',      # 60 unique
    'place_of_incorporation',         # 61 unique
    'browser_app_information',        # 10 unique
    'authentication_method',          # 5 unique
    'vpn_flag',                       # 2 unique
    'emulator_flag',                  # 2 unique

    # Risk bands
    'fis_band',                       # 5 unique
    'alert_level',                    # 5 unique
]

le = LabelEncoder()
encoded_count = 0
skipped = []

for col in encode_cols:
    if col in transactions.columns:
        transactions[col] = le.fit_transform(
            transactions[col].astype(str))
        encoded_count += 1
    else:
        skipped.append(col)

print(f"✅ Encoded  : {encoded_count} columns")
if skipped:
    print(f"⚠️  Skipped (not found): {skipped}")

print(f"\nShape after encoding : {transactions.shape}")
print()

# Check what str columns remain
remaining_str = transactions.select_dtypes(include=['str','object']).columns.tolist()
non_target    = [c for c in remaining_str if c != 'aml_typology']

if non_target:
    print("⚠️  Still str (unexpected):")
    for c in non_target:
        print(f"   {c} : {transactions[c].nunique()} unique")
else:
    print("✅ All str columns encoded")
    print("   Only 'aml_typology' remains as str (kept as M4 target)")

✅ Encoded  : 47 columns

Shape after encoding : (386570, 286)

✅ All str columns encoded
   Only 'aml_typology' remains as str (kept as M4 target)


## Step 7 — Final Verification Before Saving

In [22]:
print("=" * 50)
print("FINAL CLEAN DATASET SUMMARY")
print("=" * 50)
print(f"Shape              : {transactions.shape}")
print(f"Total missing vals : {transactions.isnull().sum().sum()}")
print(f"Duplicate rows     : {transactions.duplicated().sum()}")
print()

print("Column types after full preprocessing:")
print(transactions.dtypes.value_counts())
print()

print("Target — is_aml:")
vc = transactions['is_aml'].value_counts()
print(f"  Normal (0) : {vc[0]:,}  ({vc[0]/len(transactions)*100:.1f}%)")
print(f"  AML    (1) : {vc[1]:,}  ({vc[1]/len(transactions)*100:.1f}%)")
print()

print("Target — aml_typology:")
print(transactions['aml_typology'].value_counts())
print()

print("New features created from dates:")
date_features = ['hour', 'is_night', 'day_of_week', 'month', 'is_weekend',
                 'customer_account_age_days', 'wallet_age_days',
                 'days_since_kyc_update', 'customer_age_years']
print(transactions[date_features].describe().round(2))

FINAL CLEAN DATASET SUMMARY
Shape              : (386570, 286)
Total missing vals : 0
Duplicate rows     : 0

Column types after full preprocessing:
int64      228
float64     53
int32        3
Int64        1
str          1
Name: count, dtype: int64

Target — is_aml:
  Normal (0) : 313,716  (81.2%)
  AML    (1) : 72,854  (18.8%)

Target — aml_typology:
aml_typology
                                313716
Rapid Multi-Hop Layering         21849
High-Risk Corridor Transfer       8001
Structuring (Smurfing)            7416
Charity Abuse                     6372
Pass-Through Transit Hub          5747
Circular Transaction Loop         5468
Underground Banking (Hawala)      5163
Third-Party Payment Web           4348
Funnel Account Network            4320
Money Mule Network                4170
Name: count, dtype: int64

New features created from dates:
            hour   is_night  day_of_week      month  is_weekend  \
count  386570.00  386570.00     386570.0  386570.00   386570.00   
mean     

In [23]:
# Create output folder
output_dir = os.path.join('..', 'Data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# Save
output_file = os.path.join(output_dir, 'transactions_processed.parquet')
transactions.to_parquet(output_file, index=False)

# Confirm
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print("=" * 50)
print("✅ SAVED SUCCESSFULLY")
print("=" * 50)
print(f"  File    : {output_file}")
print(f"  Shape   : {transactions.shape}")
print(f"  Size    : {file_size_mb:.1f} MB")

✅ SAVED SUCCESSFULLY
  File    : ..\Data\processed\transactions_processed.parquet
  Shape   : (386570, 286)
  Size    : 70.7 MB


In [24]:
# ── Verify saved file ─────────────────────────────────────────────────────
verify = pd.read_parquet(os.path.join('..', 'Data', 'processed',
                                      'transactions_processed.parquet'))

print("=" * 50)
print("SAVED FILE VERIFICATION")
print("=" * 50)
print(f"Shape              : {verify.shape}")
print(f"Missing values     : {verify.isnull().sum().sum()}")
print(f"Duplicate rows     : {verify.duplicated().sum()}")
print()
print("Column types:")
print(verify.dtypes.value_counts())
print()
print("Target check:")
print(f"  is_aml values    : {verify['is_aml'].value_counts().to_dict()}")
print(f"  aml_typology     : {verify['aml_typology'].nunique()} classes")
print()
print("✅ File verified — ready for Feature Engineering!")

SAVED FILE VERIFICATION
Shape              : (386570, 286)
Missing values     : 0
Duplicate rows     : 0

Column types:
int64      228
float64     53
int32        3
Int64        1
str          1
Name: count, dtype: int64

Target check:
  is_aml values    : {0: 313716, 1: 72854}
  aml_typology     : 11 classes

✅ File verified — ready for Feature Engineering!


Done with preprocessing and ready for feature engineering